# Notebook 01: Data Exploration
EDA on raw data: missing values, revenue distributions, country analysis, product analysis, purchase timing.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

from src.data_cleaning import load_raw

df = load_raw('data/raw/Online Retail.xlsx')
print(f'Shape: {df.shape}')
print(df.dtypes)
df.head(10)

## 1. Missing Values and Data Quality

In [ ]:
# Missing values
print(df.isnull().sum())
print(f'\n% missing CustomerID: {df["CustomerID"].isna().mean()*100:.1f}%')

# Cancellations (InvoiceNo starting with 'C')
df['InvoiceNo_str'] = df['InvoiceNo'].astype(str)
cancelled = df['InvoiceNo_str'].str.upper().str.startswith('C')
print(f'Cancellation rows: {cancelled.sum():,} ({cancelled.mean()*100:.1f}%)')

## 2. Revenue and Basic Statistics

In [ ]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Quantity:', df['Quantity'].describe())
print('\nUnitPrice:', df['UnitPrice'].describe())
print('\nRevenue (line):', df['Revenue'].describe())

In [ ]:
# Monthly revenue trend
monthly = df.set_index('InvoiceDate')['Revenue'].resample('ME').sum()
fig, ax = plt.subplots(figsize=(12, 4))
monthly.plot(ax=ax)
ax.set_title('Monthly Revenue (all transactions)')
ax.set_ylabel('Revenue (£)')
plt.tight_layout()
plt.show()

## 3. Geographic Analysis

In [ ]:
# Top countries by total revenue
by_country = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(12)
by_country.plot(kind='barh', figsize=(10, 5))
plt.title('Revenue by Country (top 12)')
plt.xlabel('Revenue (£)')
plt.tight_layout()
plt.show()